The objective is to compare historical default rates across the four final borrower clusters and identify which segments have higher or lower observed default rates.

Important methodological point: Default must not be used to create the clusters. Your clustering features excluded Default, so we can now use it strictly for post-clustering analysis.

We will calculate:
1. Total borrowers in each cluster.
2. Number of defaults and non-defaults.
3. Historical default rate.
4. Share of total defaults contributed by each cluster.
5. Comparison with the overall portfolio default rate.

Step 8.1 — Default Pattern Analysis.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    r"D:\HCL_PROJECTS\Credit Portfolio Segmentation and Risk Pattern Analysis Project-3\Pickle\cleaned_credit_portfolio.pkl"
)

df = pd.read_pickle(DATA_PATH)      # not read_csv

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1347681, 15)


,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,Default,revenue_log,dti_capped,dti_extreme
0,68407277,Dec-2015,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,0,10.915107,5.91,0
1,68355089,Dec-2015,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,0,11.082158,16.06,0
2,68341763,Dec-2015,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,0,11.170449,13.85,0
3,68476807,Dec-2015,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,0,11.556311,25.37,0
4,68426831,Dec-2015,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,0,10.434145,10.20,0


Step 2 — Feature engineering and preprocessing

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Create log-transformed loan amount
df['loan_amnt_log'] = np.log1p(df['loan_amnt'])

# Clustering features (Default is deliberately excluded)
numeric_features = [
    'revenue_log',
    'loan_amnt_log',
    'fico_n',
    'dti_capped'
]

categorical_features = [
    'emp_length',
    'purpose',
    'home_ownership_n'
]

# Separate numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Fit and transform
X_cluster_scaled = preprocessor.fit_transform(df)

print("Transformed feature matrix shape:", X_cluster_scaled.shape)
print("Matrix type:", type(X_cluster_scaled))
print("Missing values:", np.isnan(X_cluster_scaled.data).sum())

Transformed feature matrix shape: (1347681, 34)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Missing values: 0


In [3]:
print(df.columns.tolist())

['id', 'issue_d', 'revenue', 'dti_n', 'loan_amnt', 'fico_n', 'experience_c', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state', 'Default', 'revenue_log', 'dti_capped', 'dti_extreme', 'loan_amnt_log']


Step 1 — Recreate the clustering pipeline

In [4]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import IncrementalPCA
from sklearn.cluster import KMeans

# Features used for clustering
numeric_features = [
    'revenue_log',
    'loan_amnt_log',
    'fico_n',
    'dti_capped'
]

categorical_features = [
    'emp_length',
    'purpose',
    'home_ownership_n'
]

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_scaled = preprocessor.fit_transform(df)

print("Transformed shape:", X_scaled.shape)

Transformed shape: (1347681, 34)


Step 2 — PCA and sampling

In [5]:
# PCA with 10 components
ipca = IncrementalPCA(
    n_components=10,
    batch_size=10000
)

batch_size = 10000

for start in range(0, X_scaled.shape[0], batch_size):
    end = min(start + batch_size, X_scaled.shape[0])
    ipca.partial_fit(X_scaled[start:end].toarray())

X_pca = np.vstack([
    ipca.transform(
        X_scaled[start:min(start + batch_size, X_scaled.shape[0])].toarray()
    )
    for start in range(0, X_scaled.shape[0], batch_size)
])

# Recreate the 5,000-row sample
rng = np.random.default_rng(42)

sample_indices = rng.choice(
    X_pca.shape[0],
    size=5000,
    replace=False
)

X_cluster_sample = X_pca[sample_indices]

# Standardize PCA features
pca_scaler = StandardScaler()
X_cluster_std = pca_scaler.fit_transform(X_cluster_sample)

print("Sample shape:", X_cluster_std.shape)

Sample shape: (5000, 10)


Step 3 — Final K-Means clustering

In [6]:
kmeans_final = KMeans(
    n_clusters=4,
    init='k-means++',
    n_init=50,
    random_state=42,
    max_iter=500
)

kmeans_labels_final = kmeans_final.fit_predict(X_cluster_std)

print("Cluster distribution:")
print(pd.Series(kmeans_labels_final).value_counts().sort_index())

Cluster distribution:
0     529
1     847
2     968
3    2656
Name: count, dtype: int64


Step 4 — Default Pattern Analysis

default_analysis = pd.DataFrame({
    'Cluster': kmeans_labels_final,
    'Default': df.iloc[sample_indices]['Default'].to_numpy()
})

default_pattern = default_analysis.groupby('Cluster').agg(
    Total_Borrowers=('Default', 'count'),
    Default_Count=('Default', 'sum')
)

default_pattern['Non_Default_Count'] = (
    default_pattern['Total_Borrowers'] -
    default_pattern['Default_Count']
)

default_pattern['Default_Rate'] = (
    default_pattern['Default_Count'] /
    default_pattern['Total_Borrowers'] * 100
)

default_pattern['Non_Default_Rate'] = (
    default_pattern['Non_Default_Count'] /
    default_pattern['Total_Borrowers'] * 100
)

overall_default_rate = default_analysis['Default'].mean() * 100

# Share of all defaults contributed by each cluster
default_pattern['Share_of_Total_Defaults'] = (
    default_pattern['Default_Count'] /
    default_pattern['Default_Count'].sum() * 100
)

# Difference from the overall default rate (percentage points)
default_pattern['Default_Rate_Difference'] = (
    default_pattern['Default_Rate'] - overall_default_rate
)

print("DEFAULT PATTERN ANALYSIS")
print(default_pattern.round(2))

print("\nOverall Default Rate:", round(overall_default_rate, 2), "%")
print("Total borrowers:", len(default_analysis))
print("Total defaults:", default_pattern['Default_Count'].sum())

1. Cluster-wise default pattern
Cluster	Borrowers	Defaults	Non-defaults	Default rate
0	529	109	420	20.60%
1	847	170	677	20.07%
2	968	171	797	17.67%
3	2,656	587	2,069	22.10%
Overall	5,000	1,037	3,963	20.74%

2. Default rate comparison
Historical default rate by cluster
Overall portfolio default rate: 20.74%


Key findings
- Cluster 3: Highest historical default rate at 22.10%, approximately 1.36 percentage points above the overall rate.
- Cluster 2: Lowest historical default rate at 17.67%, approximately 3.07 percentage points below the overall rate.
- Cluster 0: Default rate of 20.60%, close to the portfolio average.
- Cluster 1: Default rate of 20.07%, slightly below the portfolio average.
Cluster 3 also contains the largest number of borrowers (2,656) and the largest number of defaults (587).
3. Report-ready conclusion
Step 8.1: Default Pattern Analysis
Historical default patterns were analysed across the four K-Means clusters. The Default variable was excluded from the clustering process and was used only after cluster formation to evaluate the historical default distribution.

The overall default rate in the 5,000-borrower sample was 20.74%.

Cluster 3 recorded the highest default rate of 22.10%, followed by Cluster 0 at 20.60% and Cluster 1 at 20.07%. Cluster 2 exhibited the lowest default rate of 17.67%.

Cluster 3 also accounted for the largest number of historical defaults, with 587 defaults among 2,656 borrowers. These findings indicate differences in historical default patterns across the identified borrower segments.

The results are descriptive and represent historical default behaviour within the sampled dataset. They do not establish causation or represent predictions of future borrower defaults.